# 🧪 Notebook 15 — Hi-VQE

> **Module:** Quantum Computing for Computational Chemistry\
> **Level:** Advanced\
> **Estimated time:** ? min\
> **Prerequisites:** VQE, molecular Hamiltonians, computational-basis sampling, and selected-CI concepts

---

## 📌 Objective

Implements a Hi-VQE algorithm to compute the ground state of H₄.

## 🎯 Learning objectives

- Understand the basic principles behind the Hi-VQE algorithm.
- Construct and manipulate a variational ansatz.
- Optimize a quantum circuit using a classical optimizer.
- Identify relevant electronic configurations from the optimized quantum state.
- Expand the variational subspace iteratively.
- Monitor the convergence of the ground-state energy as the subspace grows..

## 🔑 Key concepts

`SQD` · `VQE` ·  `configuration selection` · `shot sampling` · `selected-CI`


# Hi-VQE: Theory and Algorithmic Workflow

## 1. Introduction

The **Hierarchical Variational Quantum Eigensolver (Hi-VQE)** is a hybrid quantum-classical algorithm designed to approximate the ground-state energy of a molecular Hamiltonian by progressively constructing and optimizing a reduced configuration space.

The central idea is to combine the strengths of quantum and classical computations:

- **Quantum computation:** Generates samples from a variational quantum state, providing information about the electronic configurations that contribute to the ground state.
- **Classical computation:** Uses the sampled configurations to construct and solve a reduced eigenvalue problem, identifying an improved approximation to the ground state.

Rather than attempting to optimize a large variational ansatz over the entire Hilbert space, Hi-VQE follows an iterative strategy in which the relevant configuration space is progressively expanded.

## 2. Basic idea

Consider a molecular Hamiltonian expressed in second quantization:

$$
\hat{H} = \sum_{pq} h_{pq} a_p^\dagger a_q
+\frac{1}{2}\sum_{pqrs} h_{pqrs}
a_p^\dagger a_q^\dagger a_s a_r.
$$

After mapping the fermionic Hamiltonian to qubits, we obtain an operator of the form

$$
H = \sum_i h_i P_i,
$$

where $P_i$ are Pauli strings.

The objective is to find its ground-state energy:

$$
E_0 = \min_{|\Psi\rangle}
\frac{\langle\Psi|H|\Psi\rangle}
{\langle\Psi|\Psi\rangle}.
$$

In Hi-VQE, we work with a restricted set of electronic configurations, which defines a subspace of the full Hilbert space:

$$
\mathcal{S} = \mathrm{span}
\left\{
|C_1\rangle, |C_2\rangle,\ldots,|C_N\rangle
\right\}.
$$

The ground state within this subspace is expressed as

$$
|\Psi_{\mathcal{S}}\rangle =
\sum_{i=1}^{N} c_i |C_i\rangle.
$$

The coefficients $c_i$ are obtained by solving the eigenvalue problem associated with the Hamiltonian projected onto this subspace:

$$
H_{\mathcal{S}} c = E c,
$$

where

$$
(H_{\mathcal{S}})_{ij}
=
\langle C_i|H|C_j\rangle.
$$

The lowest eigenvalue provides the ground-state energy approximation within the selected subspace.

The key question is therefore: **How do we identify which configurations should be included in the subspace?**

This is where the quantum component of Hi-VQE comes into play.

## 3. Algorithmic workflow

Hi-VQE addresses this problem through an iterative procedure that alternates between quantum sampling and classical diagonalization.

### Step 1. Initialize the configuration space

Start with a small set of configurations, typically containing a reference configuration such as the Hartree–Fock determinant:

$$
\mathcal{S}_0 = \{|\Psi _{\mathrm{HF}}\rangle\}.
$$

This defines the initial variational subspace.

### Step 2. Construct and optimize the quantum ansatz

Prepare a parameterized quantum state [Hint: Coupled Cluster]:

$$
|\Psi(\boldsymbol{\theta})\rangle
= U(\boldsymbol{\theta})|\Psi_{\mathrm{HF}}\rangle.
$$

Optimize its parameters by minimizing the energy expectation value:

$$
\boldsymbol{\theta}^{*}
=
\underset{\boldsymbol{\theta}}{\operatorname{argmin}}
\langle\Psi(\boldsymbol{\theta})|
H|\Psi(\boldsymbol{\theta})\rangle.
$$

The optimized quantum state provides a probability distribution over computational basis states.

### Step 3. Sample the optimized quantum state

Measure the quantum circuit in the computational basis to obtain a collection of bitstrings.

Each bitstring represents an electronic configuration (Slater determinant), and its sampling frequency provides an estimate of its probability in the optimized quantum state.

Configurations with higher probabilities are generally more relevant and become candidates for inclusion in the subspace.

### Step 4. Select new configurations

Process the measurement results to identify configurations that are not already included in the current subspace.

Select the most relevant configurations according to their sampling frequencies and add them to the configuration space:

$$
\mathcal{S}_{k+1}
=
\mathcal{S}_k \cup \mathcal{C}_{\mathrm{new}}.
$$

The selection procedure determines how quickly and efficiently the subspace grows.

### Step 5. Classical diagonalization

Construct the Hamiltonian matrix in the expanded configuration space:

$$
(H_{\mathcal{S}_{k+1}})_{ij}
=
\langle C_i|H|C_j\rangle.
$$

Solve the corresponding eigenvalue problem [Hint: SCI] to obtain the lowest eigenvalue and eigenvector:

$$
H_{\mathcal{S}_{k+1}}c
=
E_{k+1}c.
$$

This provides an improved ground-state approximation. This is the cost function for your optimization.

### Step 6. Repeat until convergence

Repeat the quantum optimization, sampling, configuration selection and classical diagonalization steps until a convergence criterion is satisfied.

For example, convergence can be monitored through the energy difference between consecutive iterations:

$$
|E_{k+1}-E_k| < \epsilon.
$$

Alternatively, the algorithm can be stopped when a maximum subspace size is reached.


## 4. Important implementation considerations

When implementing a simplified version of Hi-VQE, several aspects need to be considered:

- **Reference state:** Choose an appropriate initial electronic configuration.
- **Ansatz:** Define a parameterized circuit capable of exploring the relevant configuration space.
- **Optimization:** Select a classical optimizer to minimize the energy expectation value.
- **Sampling:** Determine the number of shots used to sample the optimized state.
- **Configuration selection:** Decide how many configurations to retain and how to handle duplicate configurations.
- **Hamiltonian matrix:** Compute the matrix elements between configurations efficiently, taking advantage of the sparsity of the electronic Hamiltonian.
- **Convergence:** Define an appropriate stopping criterion.

For simplicity, the exercise will use a relatively small molecular system and a manageable configuration space. The goal is to understand the underlying algorithmic strategy rather than to implement an optimized production-level version.

## 5. Algoritmic ideas
- Prunning: remove non-important configuration
- Alternative cost functions


In [ ]:
import numpy as np
import qiskit
import openfermion
import pyscf
from qiskit_addon_sqd.fermion import solve_fermion